# Instance Impact Framework: Full-Pipeline Case Study + LA Fire 2025 Deployment

## Objective

This notebook demonstrates the complete Instance Impact (II) pipeline for building-level disaster impact assessment:

- **Part 1**: Live end-to-end pipeline run on a Nepal flooding example (SAM3 → Stage 2a → Stage 2b)
- **Part 2**: Pre-computed results from the 2025 LA Wildfires real-world deployment (21,797 buildings, 120 cells)
- **Part 3**: Stage 1 quantitative benchmark on xView2 (building detection accuracy)
- **Part 4**: FAIR data statement and reproducibility guide


## Reproducibility and Environment Strategy

This notebook is designed for top-down execution in managed research kernels where direct modification of the base environment may be restricted. To preserve reproducibility, dependencies are installed into a project-local directory (`.nb_vendor`) and injected into both notebook imports and subprocess-based stage scripts.

This strategy provides two practical benefits: (1) compatibility across CPU-only and GPU-enabled sessions on i-guide, and (2) isolation from user-site package drift that can otherwise break scientific dependencies. The next two code cells perform dependency bootstrap and import validation before any model stage begins.

## Methods: Execution Protocol

The pipeline is implemented as a staged experimental protocol where each stage emits explicit artifacts used by subsequent stages. The initial setup cells establish path conventions, dependency resolution, authentication, and runtime device configuration so that Stage 1 through Stage 2b can be executed and audited consistently.

Methodologically, this notebook follows a deterministic order: environment bootstrap -> stage execution -> stage-level sanity reporting -> final synthesis and uncertainty-driven interpretation.

## Model Specification (Run Configuration)

This case study uses the packaged inference stack exactly as configured in this notebook run.

- **Stage 1 (Instance Extraction):** SAM3 via `samgeo` + `transformers` backend, producing instance-level polygons and confidence from pre-disaster imagery.
- **Stage 2a (Exposure Proxy):** EfficientNet-B0-based 4-channel model (RGB + mask) for population-regression signal and building-type classification.
- **Stage 2b (Damage + Uncertainty):** Siamese ordinal CORAL-style damage inference, executed as a calibrated multi-checkpoint ensemble with uncertainty metrics (`pmax`, margin, entropy, expected-severity variance).
- **Calibration mode:** temperature scaling (packaged calibration directories applied during Stage 2b ensemble inference).
- **Checkpoint provenance:** model files are loaded from `II_package/models/` with configs in `II_package/configs/` and calibration artifacts in `II_package/calibration/`.

This section is intentionally concise; implementation-level details remain traceable in the stage scripts and companion technical logs.

In [ ]:
from pathlib import Path
import os
import json
import shutil
import subprocess
import shlex
import sys
import importlib

# Resolve notebook directory robustly.
# Managed kernels (e.g., i-guide geoai) may have a stale or deleted cwd,
# so we prefer IPython's _dh (notebook launch dir) with cwd() as fallback.
try:
    _nb_dir = Path(globals()["_dh"][0])
except Exception:
    try:
        _nb_dir = Path.cwd()
    except Exception:
        _nb_dir = Path(os.environ.get("HOME", "/tmp"))

VENDOR = _nb_dir / ".nb_vendor"
VENDOR.mkdir(parents=True, exist_ok=True)

if str(VENDOR) not in sys.path:
    sys.path.insert(0, str(VENDOR))
importlib.invalidate_caches()
print("Using local vendor path:", VENDOR)
print("Base imports ready.")

### Core Imports and Local Path Setup

Foundational Python modules are imported and the `.nb_vendor` overlay path is initialized. If `.nb_vendor` exists, it is prioritized in `sys.path` to ensure deterministic dependency resolution for this notebook run.

In [ ]:
# Cell 1/2: detect-then-install notebook-local dependencies into .nb_vendor
#
# For packages where the base kernel has an older version that is insufficient,
# we check for the specific symbol or minimum version needed — not just importability.

if str(VENDOR) not in sys.path:
    sys.path.insert(0, str(VENDOR))
importlib.invalidate_caches()


def pip_install(*pkgs, no_deps=True):
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "--target", str(VENDOR)]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(pkgs)
    print("[cmd]", " ".join(cmd))
    subprocess.run(cmd, check=True)


def check_import(module_name):
    importlib.invalidate_caches()
    importlib.import_module(module_name)


def check_attr(module_name, attr_name):
    """Check that a module exposes a specific attribute (version-sensitive)."""
    importlib.invalidate_caches()
    mod = importlib.import_module(module_name)
    if not hasattr(mod, attr_name):
        raise ImportError(f"{module_name} missing '{attr_name}' (version too old: {getattr(mod, '__version__', '?')})")


def check_min_version(module_name, min_ver_tuple):
    """Check that module.__version__ >= min_ver_tuple, e.g. (2025, 10, 22)."""
    importlib.invalidate_caches()
    mod = importlib.import_module(module_name)
    ver_str = getattr(mod, "__version__", "0")
    ver_parts = tuple(int(x) for x in ver_str.split(".")[:len(min_ver_tuple)])
    if ver_parts < min_ver_tuple:
        raise ImportError(f"{module_name} {ver_str} < {'.'.join(map(str, min_ver_tuple))}")


def check_sam3model():
    importlib.invalidate_caches()
    from transformers import Sam3Model  # noqa: F401


def check_samgeo3():
    importlib.invalidate_caches()
    from samgeo import SamGeo3  # noqa: F401


def purge_modules(*prefixes):
    """Remove cached module entries so the next import picks up .nb_vendor.
    Only safe for pure-Python packages; native extensions (PyO3/Rust) like
    tokenizers, regex, safetensors cannot be re-initialized after unloading."""
    stale = [k for k in sys.modules if any(k == p or k.startswith(p + ".") for p in prefixes)]
    for k in stale:
        del sys.modules[k]
    importlib.invalidate_caches()


def ensure(pkg_spec, check_fn):
    try:
        check_fn()
        print(f"[skip] {pkg_spec} (already importable)")
        return "skipped"
    except Exception as e:
        print(f"[need] {pkg_spec} ({type(e).__name__}: {e})")
        pip_install(pkg_spec)
        importlib.invalidate_caches()
        return "installed"


results = []

# Core runtime packages (install only when checks fail).
results.append(("timm==1.0.25", ensure("timm==1.0.25", lambda: check_import("timm"))))
results.append(("opencv-python-headless==4.10.0.84", ensure("opencv-python-headless==4.10.0.84", lambda: check_import("cv2"))))
results.append(("segment-anything", ensure("segment-anything", lambda: check_import("segment_anything"))))
results.append(("segment-geospatial", ensure("segment-geospatial", check_samgeo3)))
results.append(("geoai-py", ensure("geoai-py", lambda: check_import("geoai"))))
results.append(("geopandas", ensure("geopandas", lambda: check_import("geopandas"))))

# Sam3Model support from transformers main.
results.append(("git+https://github.com/huggingface/transformers.git", ensure("git+https://github.com/huggingface/transformers.git", check_sam3model)))

# Explicit transitive/runtime deps — version-sensitive checks where base kernel is too old.
results.append(("httpx", ensure("httpx", lambda: check_import("httpx"))))
results.append(("httpcore", ensure("httpcore", lambda: check_import("httpcore"))))
results.append(("huggingface_hub>=0.30", ensure("huggingface_hub>=0.30", lambda: check_attr("huggingface_hub", "is_offline_mode"))))
results.append(("regex>=2025.10.22", ensure("regex>=2025.10.22", lambda: check_min_version("regex", (2025, 10, 22)))))
results.append(("tokenizers", ensure("tokenizers", lambda: check_min_version("tokenizers", (0, 20)))))
results.append(("safetensors", ensure("safetensors", lambda: check_min_version("safetensors", (0, 5)))))
results.append(("h11", ensure("h11", lambda: check_import("h11"))))
results.append(("anyio", ensure("anyio", lambda: check_import("anyio"))))
results.append(("sniffio", ensure("sniffio", lambda: check_import("sniffio"))))
results.append(("idna", ensure("idna", lambda: check_import("idna"))))
results.append(("certifi", ensure("certifi", lambda: check_import("certifi"))))

installed = sum(1 for _, s in results if s == "installed")
skipped = sum(1 for _, s in results if s == "skipped")

# Purge stale sys.modules entries for all packages that were installed into .nb_vendor.
# This ensures the NEXT cell's imports resolve from .nb_vendor, not from cached base-kernel modules.
if installed > 0:
    # Only purge pure-Python packages. Native extensions (tokenizers, regex,
    # safetensors) use PyO3/Rust and cannot be re-initialized after unloading.
    purge_modules(
        "transformers", "huggingface_hub",
        "samgeo", "geoai", "geopandas", "segment_anything",
        "timm", "cv2", "httpx", "httpcore",
        "h11", "anyio", "sniffio", "idna", "certifi",
    )
    print(f"Purged stale module cache for {installed} newly installed packages.")

print(f"Dependency pass complete. installed={installed}, skipped={skipped}, vendor={VENDOR}")

### Dependency Bootstrap

Notebook-local **detect-then-install** dependency setup targets `.nb_vendor`. Packages that are already importable with sufficient version are skipped, reducing rerun time while preserving reproducibility.

In [ ]:
# Cell 2/2: validate imports from .nb_vendor
#
# Purge + re-import to guarantee we pick up .nb_vendor versions,
# not stale base-kernel modules cached in sys.modules during detection.

if str(VENDOR) not in sys.path:
    sys.path.insert(0, str(VENDOR))

# Only purge pure-Python packages; native extensions (tokenizers, regex,
# safetensors) use PyO3 and cannot be re-initialized in the same process.
purge_modules("transformers", "huggingface_hub",
              "samgeo", "geoai", "timm")

import timm
from transformers import Sam3Model
from samgeo import SamGeo3

print("Using vendor path:", VENDOR)
print("timm:", timm.__version__)
print("transformers:", __import__("transformers").__version__)
print("Sam3Model: OK")
print("SamGeo3: OK")

### Dependency Validation

Critical imports (`timm`, `Sam3Model`, `SamGeo3`) are validated from the local environment before any model stage is executed.

In [ ]:
# --- Token setup for Stage1 SAM3 access ---
# Stage 1 uses SAM3, a gated model on Hugging Face that requires authentication.
# To obtain a token: https://huggingface.co/settings/tokens (read-only is sufficient).
# You must also accept the SAM3 model license at https://huggingface.co/facebook/sam3.
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN", "").strip():
    token = getpass("Enter HF_TOKEN (input hidden, leave blank to skip): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN set for current kernel session.")
    else:
        print("HF_TOKEN not provided. Stage 1 may fail if SAM3 model access is required.")
else:
    print("HF_TOKEN already present in environment.")

### Model Access Token

Stage 1 requires authenticated access to SAM3, a gated model hosted on Hugging Face. This cell prompts for `HF_TOKEN` if one is not already set in the environment. A read-only token is sufficient. If you do not have a token, create one at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and accept the model license at [huggingface.co/facebook/sam3](https://huggingface.co/facebook/sam3).

In [ ]:
# --- 1) Study configuration: full pipeline run ---

# Auto-locate package root using the same robust notebook-dir as VENDOR.
PACKAGE_ROOT = _nb_dir
if not (PACKAGE_ROOT / "run_pipeline.sh").exists() and (PACKAGE_ROOT / "II_package" / "run_pipeline.sh").exists():
    PACKAGE_ROOT = PACKAGE_ROOT / "II_package"

# Input image pair (customize these two paths).
PRE_IMAGE = PACKAGE_ROOT / "example_image_pair" / "nepal-flooding_00000408_pre_disaster.png"
POST_IMAGE = PACKAGE_ROOT / "example_image_pair" / "nepal-flooding_00000408_post_disaster.png"

# Run identifier for outputs.
RUN_ID = "nb_full_pipeline_nepal_flooding_00000408"

# Runtime settings (aligned with package defaults).
PYTHON_BIN = sys.executable
CUDA_VISIBLE_DEVICES = "0"
VERBOSE_RUN = False

print("PACKAGE_ROOT:", PACKAGE_ROOT)
print("PRE_IMAGE:", PRE_IMAGE)
print("POST_IMAGE:", POST_IMAGE)
print("RUN_ID:", RUN_ID)
print("PYTHON_BIN:", PYTHON_BIN)

### Experiment Configuration

Case-study inputs and runtime identifiers (`PACKAGE_ROOT`, pre/post image paths, `RUN_ID`, Python executable) are defined here. These parameters govern artifact locations and reproducible reruns.

### Git LFS Checkpoint Verification

Model checkpoint files (`.pt`) are stored via Git LFS. After a fresh `git clone`, they may appear as small pointer stubs (~130 bytes) instead of actual binaries. This cell detects LFS pointers and attempts to pull the real files automatically, installing `git-lfs` via conda if it is not already available.

In [ ]:
# Detect Git LFS pointer files in model checkpoints and auto-pull if possible.

_lfs_pointers = []
for _sub in ["models/stage2a", "models/stage2b", "calibration"]:
    _dir = PACKAGE_ROOT / _sub
    if not _dir.exists():
        continue
    for _f in _dir.rglob("*.pt"):
        if _f.stat().st_size < 4096:
            _lfs_pointers.append(_f)
    for _f in _dir.rglob("*.npz"):
        if _f.stat().st_size < 4096:
            _lfs_pointers.append(_f)

if _lfs_pointers:
    print(f"Detected {len(_lfs_pointers)} Git LFS pointer file(s):")
    for _f in _lfs_pointers[:5]:
        print(f"  {_f.relative_to(PACKAGE_ROOT)}  ({_f.stat().st_size} bytes)")
    if len(_lfs_pointers) > 5:
        print(f"  ... and {len(_lfs_pointers) - 5} more")

    _repo_root = PACKAGE_ROOT.parent

    # Ensure git-lfs is available; install via conda if missing.
    try:
        subprocess.run(["git", "lfs", "version"], check=True, capture_output=True)
        print("git-lfs: already installed.")
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("git-lfs not found. Installing via conda...")
        subprocess.run(
            ["conda", "install", "-c", "conda-forge", "-y", "git-lfs"],
            check=True,
        )
        subprocess.run(["git", "lfs", "version"], check=True)
        print("git-lfs installed successfully.")

    print(f"Running 'git lfs pull' in {_repo_root} ...")
    subprocess.run(["git", "lfs", "install"], check=True, cwd=str(_repo_root), capture_output=True)
    subprocess.run(["git", "lfs", "pull"], check=True, cwd=str(_repo_root))
    _still_bad = [f for f in _lfs_pointers if f.stat().st_size < 4096]
    if _still_bad:
        raise RuntimeError(f"{len(_still_bad)} files still appear to be LFS pointers after pull.")
    print("Git LFS pull complete. All model checkpoints are now real files.")
else:
    print("Model checkpoints OK (no LFS pointers detected).")

In [ ]:
# --- 2) Prepare a clean run directory ---
# This cell keeps runtime setup deterministic by resetting the run folder.

RUN_ROOT = PACKAGE_ROOT / "outputs" / "driver_runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
print("RUN_DIR (fresh):", RUN_DIR)

### Output Workspace Reset

A fresh run directory is created for the current `RUN_ID`, ensuring that outputs correspond only to the present execution and not stale artifacts from prior runs.

In [ ]:
# --- 2b) Scientific/runtime imports + device selection ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

STAGE1_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
STAGE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("numpy:", np.__version__)
print("numpy path:", np.__file__)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available(), "count:", torch.cuda.device_count())
print("STAGE1_DEVICE:", STAGE1_DEVICE)
print("STAGE_DEVICE:", STAGE_DEVICE)
print("Scientific imports ready.")

## Stage 1: Building Instance Extraction

Stage 1 detects and delineates building instances from the pre-disaster image, then exports instance-level polygons and confidence scores used as anchors for downstream modeling. This stage defines the shared unit of analysis (`instance_id`) for the full framework.

Expected logs include `HF_TOKEN` notices, SamGeo package hints, and raster non-georeferenced warnings; these are normal for this PNG/pixel-space workflow. On CPU-only sessions, Stage 1 commonly requires approximately 8-15 minutes for one image under this notebook configuration.

In [ ]:
# --- 3a) Stage 1: execute ---

env = dict(os.environ)
env["PYTHON_BIN"] = PYTHON_BIN
env["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
# Ensure stage subprocesses can resolve notebook-local vendor packages.
if VENDOR.exists():
    existing_pp = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(VENDOR) + ((":" + existing_pp) if existing_pp else "")

def run_step(cmd, title):
    print(f"\n===== {title} =====")
    print("[cmd]", " ".join(shlex.quote(str(x)) for x in cmd))
    subprocess.run([str(x) for x in cmd], check=True, cwd=str(PACKAGE_ROOT), env=env)

tile_id = PRE_IMAGE.stem
if tile_id.endswith("_pre_disaster"):
    tile_id = tile_id[: -len("_pre_disaster")]

pair_dir = RUN_DIR / "pair_inputs"
pair_dir.mkdir(parents=True, exist_ok=True)
pre_link = pair_dir / f"{tile_id}_pre_disaster.png"
post_link = pair_dir / f"{tile_id}_post_disaster.png"
for dst in [pre_link, post_link]:
    if dst.exists() or dst.is_symlink():
        dst.unlink()
pre_link.symlink_to(PRE_IMAGE.resolve())
post_link.symlink_to(POST_IMAGE.resolve())

stage1_out = RUN_DIR / "stage1"
shared_out = RUN_DIR / "shared_instances_r48"
stage2a_input_csv = RUN_DIR / "stage2a_infer_input.csv"
stage2a_out_csv = RUN_DIR / "stage2a_predictions.csv"
stage2b_out_jsonl = RUN_DIR / "stage2b_ensemble.jsonl"
presented_csv = RUN_DIR / "instance_results_presented.csv"
uncertain_csv = RUN_DIR / "instance_results_top_uncertain.csv"
vis_dir = RUN_DIR / "vis_instance_level"

run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "stage1/SAM3_Final_20260226/scripts/run_sam3_building_infer.py",
        "--input", pair_dir,
        "--output", stage1_out,
        "--pattern", f"{tile_id}_pre_disaster.png",
        "--max-images", "1",
        "--prompt", "building",
        "--min-size", "30",
        "--output-style", "notebook",
        "--batch-size", "1",
        "--device", STAGE1_DEVICE,
        "--backend", "transformers",
        "--tile-size", "512",
        "--overlap", "64",
    ],
    "Stage 1: Building Instance Extraction",
)

label_dir = stage1_out / "labels"

### Stage 1 Sanity Report

Stage 1 completion is confirmed by checking expected summary artifacts and counting generated instance-level label outputs.

In [ ]:
# --- 3b) Stage 1: report ---
stage1_summary = stage1_out / "run_summary.json"
print("[stage1] run_summary_exists=", stage1_summary.exists())
print("[stage1] label_json_count=", len(list(label_dir.glob("*_prediction.json"))) )

### Result Discussion: Stage 1 Outputs

Stage 1 completed successfully and produced instance-level outputs required by the remainder of the pipeline. The artifact checks in this section confirm that label JSON generation and summary logging are present and internally consistent.

### Shared Instance Artifact Generation

Stage 1 outputs are transformed into standardized per-instance pre/post crops and masks (256×256, with mask M and ring R) used by both Stage 2a and Stage 2b.

In [ ]:
# --- 4a) Shared subimages: execute ---
run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/generate_shared_instance_subimages.py",
        "--stage1_labels_dir", label_dir,
        "--pre_images_dir", pair_dir,
        "--post_images_dir", pair_dir,
        "--out_root", shared_out,
        "--crop_size", "256",
        "--ring_radius_px", "48",
        "--strict_images",
        "--num_workers", "4",
        "--chunk_size", "100",
        "--log_every", "50",
    ],
    "Shared Subimage Generation",
)

shared_csv = shared_out / "shared_instance_samples.csv"

# Expected runtime: ~150 seconds

## Data and Split Protocol (Notebook Scope vs. Training Scope)

This notebook is a **single-pair inference case study** designed to demonstrate end-to-end execution and interpretation. It does not perform model training or re-estimate train/validation/test metrics.

- **Notebook scope:** pre/post pair -> instance extraction -> shared artifacts -> Stage 2a/2b inference -> synthesis and visualization.
- **Artifact schema:** instance-linked rows in `shared_instance_samples.csv` connect Stage 1 geometry/confidence with Stage 2a and Stage 2b outputs.
- **Split methodology context:** tile-blocked training/evaluation design and broader model-development protocols are documented in `Stage2b.md` and associated training scripts.

Interpret this notebook's results as case-level operational evidence, not as a replacement for aggregate benchmark reporting.

## AI-Ready Data and Metadata

The shared artifact stage converts Stage 1 outputs into a standardized instance table (`shared_instance_samples.csv`) that links each building to pre/post crops and binary masks. This table serves as the common input contract for both Stage 2a and Stage 2b, ensuring geometric and identity consistency across downstream models.

Metadata propagated through the pipeline includes tile and event identifiers, per-instance unique identifiers, and Stage 1 segmentation confidence scores. These are augmented progressively by Stage 2a exposure outputs and Stage 2b calibrated uncertainty metrics as the pipeline advances.

During artifact generation, instances with malformed polygon geometries (reported as `bad_wkt` exclusions) are filtered out. This is an intentional quality gate: skipping geometrically invalid instances improves downstream robustness at the cost of minor coverage reduction.

In [ ]:
# --- 4b) Shared subimages: report ---
df_shared = pd.read_csv(shared_csv)
print("[shared] rows=", len(df_shared))
if "sam3_confidence" in df_shared.columns:
    x = pd.to_numeric(df_shared["sam3_confidence"], errors="coerce").dropna()
    if len(x) > 0:
        print("[shared] sam3_confidence median=", round(float(np.median(x)), 4))

## Stage 2: Exposure and Damage Inference

Stage 2a estimates exposure-related signals (building type and population proxy) from shared instance crops, while Stage 2b predicts calibrated damage state and uncertainty from paired pre/post evidence. Together, these components provide complementary views of potential impact: who/what may be exposed and how severely structures appear affected.

In [ ]:
# --- 5a) Stage 2a: execute ---
run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/build_stage2a_infer_csv.py",
        "--shared_csv", shared_csv,
        "--out_csv", stage2a_input_csv,
        "--crop_col", "pre_crop",
        "--mask_col", "mask_M",
    ],
    "Stage 2a: Build Inference CSV",
)

run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/infer_stage2a.py",
        "--input_csv", stage2a_input_csv,
        "--ckpt", PACKAGE_ROOT / "models/stage2a/stage2a_best_model.pt",
        "--out_csv", stage2a_out_csv,
        "--batch_size", "4",
        "--num_workers", "4",
        "--device", STAGE_DEVICE,
        "--print_examples", "5",
    ],
    "Stage 2a: Population/Type Inference",
)

df_s2a = pd.read_csv(stage2a_out_csv)

### Stage 2a Summary Statistics

Stage 2a output coverage and compact distribution diagnostics (building-type counts and population-proxy quantiles) are reported below.

In [ ]:
# --- 5b) Stage 2a: report ---
print("[stage2a] rows=", len(df_s2a))
if "pred_type_class" in df_s2a.columns:
    print("[stage2a] type_counts=", df_s2a["pred_type_class"].value_counts().to_dict())
if "pred_population" in df_s2a.columns:
    p = pd.to_numeric(df_s2a["pred_population"], errors="coerce").dropna()
    if len(p) > 0:
        q = np.percentile(p, [50, 90, 99])
        print(f"[stage2a] population p50/p90/p99 = {q[0]:.2f}/{q[1]:.2f}/{q[2]:.2f}")

Stage 2a completed for all generated instances and returned a concentrated class mix with broad population-proxy spread. This suggests the model is producing differentiated exposure estimates rather than a narrow constant output.

Because Stage 2a is used here as an exposure signal, these values are best interpreted comparatively within the same run (higher vs. lower relative exposure) rather than as census-grade absolute counts.

### Stage 2b Ensemble Execution

Calibrated ensemble damage inference is executed below, computing per-instance uncertainty metrics (`pmax`, margin, entropy, variance of expected severity).

In [ ]:
# --- 6a) Stage 2b: execute ---
run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/infer_stage2_ensemble.py",
        "--csv", shared_csv,
        "--ckpts", "models/stage2b/inference0.7273.pt,models/stage2b/inference0.7066_seed9999.pt,models/stage2b/inference0.7034_seed7777.pt",
        "--configs", "configs/stage2b/run019_seed2025_train_config.json,configs/stage2b/seed9999_train_config.json,configs/stage2b/seed7777_train_config.json",
        "--weights", "4,3,2",
        "--calibration_dirs", "calibration/calibration_run019_r48,calibration/calibration_seed9999_r48,calibration/calibration_seed7777_r48",
        "--calibration_method", "temperature",
        "--out_jsonl", stage2b_out_jsonl,
        "--batch_size", "2",
        "--num_workers", "4",
        "--device", STAGE_DEVICE,
        "--print_examples", "5",
        "--log_every_steps", "20",
    ],
    "Stage 2b: Damage/Uncertainty Inference",
)

rows_s2b = []
with stage2b_out_jsonl.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows_s2b.append(json.loads(line))

### Stage 2b Sanity Report

Quick diagnostics on Stage 2b outputs are provided below, including row counts, predicted damage-class distribution, and aggregate confidence behavior.

In [ ]:
# --- 6b) Stage 2b: report ---
print("[stage2b] rows_jsonl=", len(rows_s2b))
if rows_s2b:
    y_counts = pd.Series([str(r.get("y_pred_ensemble", "")) for r in rows_s2b]).value_counts().to_dict()
    print("[stage2b] damage_pred_counts=", y_counts)
    pmax_vals = [float(r.get("pmax", np.nan)) for r in rows_s2b]
    pmax_vals = [v for v in pmax_vals if np.isfinite(v)]
    if pmax_vals:
        print("[stage2b] pmax mean=", round(float(np.mean(pmax_vals)), 4))

Stage 2b produced full-row predictions with a dominant share in one damage class and a strong average `pmax`, indicating confident classification for many instances in this tile. At the same time, sample-level entropy and variance outputs show non-trivial uncertainty in selected cases.

This dual behavior is expected and desirable: confident predictions for clear examples, alongside explicit uncertainty signals for ambiguous instances.

## Stage 3: Instance-Level Synthesis and Interpretation

This stage merges Stage 1, Stage 2a, and Stage 2b outputs by shared instance identity, producing a unified instance table and uncertainty-ranked cases for review. Visualization overlays link quantitative predictions back to local image evidence, improving interpretability and supporting human-in-the-loop triage.

In [ ]:
# --- 7a) Synthesis + visualization: execute ---
run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/present_instance_results.py",
        "--shared_csv", shared_csv,
        "--stage2a_csv", stage2a_out_csv,
        "--stage2b_jsonl", stage2b_out_jsonl,
        "--out_csv", presented_csv,
        "--out_top_uncertain_csv", uncertain_csv,
        "--top_k_uncertain", "30",
        "--print_top_n", "15",
    ],
    "Synthesis: Instance-Level Presentation",
)

run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/visualize_stage2_overlays.py",
        "--pred_jsonl", stage2b_out_jsonl,
        "--csv", shared_csv,
        "--stage2a_csv", stage2a_out_csv,
        "--out_dir", vis_dir,
        "--max_outputs", "100",
        "--fill_opacity", "0.5",
    ],
    "Visualization: Instance Overlays",
)

print("\n[done] run_dir:", RUN_DIR)
print("[done] stage1_out:", stage1_out)
print("[done] shared_csv:", shared_csv)
print("[done] stage2a_out:", stage2a_out_csv)
print("[done] stage2b_out:", stage2b_out_jsonl)
print("[done] presented_csv:", presented_csv)
print("[done] top_uncertain_csv:", uncertain_csv)
print("[done] vis_dir:", vis_dir)

## Output Interpretation and Responsible Use

The synthesis stage produces an instance-level table suitable for prioritization (e.g., high damage with high exposure) and an uncertainty-ranked list for human review. Overlay visualizations provide localized evidence to support interpretability.

Predictions should be treated as decision-support signals rather than definitive truth, especially under distribution shift or low-confidence conditions. Operational usage should retain human-in-the-loop validation for high-stakes decisions.

In [ ]:
# --- 7b) Synthesis: report/load ---
df = pd.read_csv(presented_csv)
df_unc = pd.read_csv(uncertain_csv)
print("rows presented:", len(df))
print("rows uncertain:", len(df_unc))

### Consolidated Summary Table

Final joined counts and high-level categorical summaries across all pipeline stages are printed below for rapid interpretation.

In [ ]:
# --- 8) Final synthesis summary ---
print("[summary] all_rows=", len(df))
if "has_stage2a" in df.columns and "has_stage2b" in df.columns:
    joined = int((df["has_stage2a"].astype(bool) & df["has_stage2b"].astype(bool)).sum())
    print("[summary] all_three_joined=", joined)

if "stage2a_pred_type_class" in df.columns:
    print("[summary] counts_by_stage2a_type=", df["stage2a_pred_type_class"].fillna("").value_counts().to_dict())
if "stage2b_pred_damage_class" in df.columns:
    d = df["stage2b_pred_damage_class"].astype(str)
    print("[summary] counts_by_stage2b_damage_class_valid_only=", d[d != ""].value_counts().to_dict())

### Result Discussion: Consolidated Synthesis

The final joined summary indicates complete cross-stage linkage for this run (`all_three_joined` matches total rows), which confirms end-to-end wiring integrity. The reported counts and quantiles should be interpreted as case-specific evidence, not global performance claims.

Operationally, these summaries are most useful for relative prioritization within this tile: combining Stage 2a exposure signals with Stage 2b severity and uncertainty to identify high-impact or high-review candidates.

### Distributional Diagnostics

Key distributions (damage classes, confidence, exposure proxy) are visualized below to support quality checks and comparative reporting.

In [ ]:
# --- 9) Distributional diagnostics ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if "stage2b_pred_damage_class" in df.columns:
    damage = df["stage2b_pred_damage_class"].astype(str)
    damage = damage[damage != ""]
    damage.value_counts().sort_index().plot(kind="bar", ax=axes[0], title="Damage Class Counts")
    axes[0].set_xlabel("damage class")

if "stage2b_pmax" in df.columns:
    pd.to_numeric(df["stage2b_pmax"], errors="coerce").dropna().plot(kind="hist", bins=20, ax=axes[1], title="Stage2b Confidence (pmax)")
    axes[1].set_xlabel("pmax")

if "stage2a_pred_population" in df.columns:
    pd.to_numeric(df["stage2a_pred_population"], errors="coerce").dropna().plot(kind="hist", bins=20, ax=axes[2], title="Stage2a Predicted Population")
    axes[2].set_xlabel("population")

plt.tight_layout()
plt.show()

### Result Discussion: Distribution Diagnostics

The distribution plots provide a quick structural sanity check for this case study. Damage predictions are concentrated in a subset of classes, confidence values are generally high but not saturated, and Stage 2a exposure estimates span a broad range.

This combination is consistent with a workable inference run: the model is not collapsing to a single confidence value, while still identifying a dominant damage pattern for this specific tile.

### Uncertainty-Centered Case Review

Top uncertain instances and sample overlays are presented below, linking numeric uncertainty indicators back to localized image evidence.

## Operational Interpretation Rules (Action-Oriented)

The synthesis outputs are intended for triage-style decision support. A practical rule set for this notebook's outputs is:

- **High impact + high confidence** (high severity/exposure, high `pmax`, low entropy): act early; prioritize response.
- **High impact + high uncertainty** (high severity/exposure, low margin/high entropy/variance): prioritize verification first, then action.
- **Low impact + low confidence**: defer immediate intervention, monitor, and re-check with additional context.
- **Cross-stage disagreement** (weak Stage 1 or Stage 2a confidence plus uncertain Stage 2b): route to analyst review queue.

This rule set is a workflow guide, not a substitute for policy judgment; high-stakes usage should retain human oversight.

In [ ]:
# --- 10) Uncertainty table + overlay preview ---
preview_cols = [
    "instance_id",
    "tile_id",
    "stage2b_pred_damage_class",
    "stage2b_pmax",
    "stage2b_entropy",
    "stage2b_var_expected_severity_weighted",
    "stage1_sam3_confidence",
    "stage2a_pred_population",
    "stage2a_pred_type_class",
    "stage2a_pred_type_conf",
]
cols = [c for c in preview_cols if c in df_unc.columns]
print("Top uncertain instances (first 20):")
display(df_unc[cols].head(20))

all_png = sorted(vis_dir.glob("*.png"))
assert len(all_png) > 0, f"No overlay images found in {vis_dir}"
img_paths = all_png[:6]
print("overlay files found:", len(all_png))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for ax, p in zip(axes, img_paths):
    im = Image.open(p)
    ax.imshow(im)
    ax.set_title(p.name[:56], fontsize=8)
    ax.axis("off")
for ax in axes[len(img_paths):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Instance-Level Interpretation of Predictions

Each panel should be read as a joint evidence unit: `pred` and `exp severity` indicate damage level, `pmax`/`margin`/`entropy` indicate confidence structure, `pop` represents exposure (Stage 2a), and `s1_conf`/`s2a_conf` provide upstream reliability context.

In the first row, **top-left** (`pred=2`, `exp severity=1.945`, `pmax=0.894`, `margin=0.855`, `entropy=0.453`, `pop=416`) is a high-confidence major-damage case with moderate exposure, making it a **Priority 1** candidate because inundation and contextual cues appear coherent. **Top-middle** (`pred=1`, `exp severity=0.337`, `pmax=0.624`, `margin=0.336`, `entropy=0.925`, `pop=116`) shows low severity, low exposure, and high ambiguity, so it fits **Priority 3-4** (low impact, verify only if resources allow). **Top-right** (`pred=2`, `exp severity=1.796`, `pmax=0.764`, `margin=0.616`, `entropy=0.763`, `pop=2682`) combines high exposure and strong damage under moderate confidence, placing it in **Priority 1-2**.

In the second row, **bottom-left** (`pred=1`, `exp severity=1.049`, `pmax=0.893`, `margin=0.841`, `entropy=0.436`, `pop=724`) is a borderline minor/major case but with high-confidence minor assignment, so it is best treated as **mid priority**. **Bottom-middle** (`pred=2`, `exp severity=1.599`, `pmax=0.642`, `margin=0.386`, `entropy=0.915`, `pop=1786`) has high impact potential but substantial uncertainty; this is a **Priority 2** case that should be verified before decisive action. **Bottom-right** (`pred=2`, `exp severity=1.928`, `pmax=0.933`, `margin=0.899`, `entropy=0.314`, `pop=3013`) is the strongest high-impact profile in the sample (very high exposure + very confident major damage), making it a clear **Priority 1** instance.

Across panels, four patterns are consistent: (1) impact is driven by **exposure x damage**, not damage alone; (2) high-entropy/low-margin predictions are useful flags for analyst review rather than model failure; (3) neighborhood context appears to contribute meaningfully in dense flooded clusters; and (4) exposure variation (~100 to ~3000 in this sample) materially changes triage priority even when predicted damage class is similar.

Overall, the outputs support a practical decision rule: use damage-exposure combination to rank consequence, then use uncertainty to decide whether to act immediately or verify first.

## Conclusion

### Impact Potential and Real-World Use Cases
This notebook delivers a complete building-instance disaster impact workflow that can support policy, operations, and research. By combining instance extraction, exposure proxy estimation, and calibrated damage uncertainty, the framework moves beyond coarse maps toward decision-support-ready outputs.

**Informing Disaster Risk Reduction and Response**
Instance-level damage and uncertainty can guide triage workflows by prioritizing locations with both high predicted severity and high confidence, while routing ambiguous cases to human review. This supports more transparent and auditable emergency decision pipelines.

**Targeted Infrastructure and Resilience Planning**
Merged outputs can identify building clusters where potential damage and exposure are both elevated, helping planners prioritize inspections, mitigation investments, and resilience interventions at finer spatial scales than area-level summaries.

**Socioeconomic and Humanitarian Targeting**
Although Stage 2a outputs are proxy estimates, they provide a practical signal for exposure-aware prioritization when merged with damage predictions. This can support NGOs and public agencies in resource allocation when rapid, building-level situational awareness is needed.

**Operational AI with Interpretable Evidence**
The uncertainty-ranked table and overlay products provide direct visual context for each prediction, improving interpretability and making model outputs more usable in analyst-in-the-loop environments.

### Next Steps and Potential Improvements
- Expand multi-event and multi-region validation to quantify generalization and domain-shift behavior more rigorously.
- Improve Stage 2a robustness by recalibration or region-adaptive fine-tuning where representative labels are available.
- Add explicit quality-control checks for artifact integrity (e.g., checkpoint/LFS validation, schema checks) as first-class notebook diagnostics.
- Introduce uncertainty-aware decision thresholds and abstention policies tailored to specific operational contexts.
- Evaluate lightweight acceleration options for Stage 1 on CPU-constrained sessions (tiling strategy, caching, or staged reuse).
- Extend synthesis outputs into downstream policy/operations dashboards for repeatable cross-event comparison.

### Limitations
Current results are model-derived estimates and should not be treated as ground truth without contextual validation. Performance may vary by geography, hazard characteristics, and imaging conditions; therefore, high-stakes usage should retain human oversight and uncertainty-aware review.

For comparative studies, rerun this workflow across multiple tiles/events and summarize aggregated uncertainty-aware metrics rather than relying on a single case study.

---

# Part 2 — Real-World Deployment: 2025 LA Wildfires

This section presents **pre-computed results** from applying the full Instance Impact pipeline to the 2025 Southern California Wildfires — a real operational deployment at scale.

| Attribute | Value |
|---|---|
| **Area** | ~197 km² across Los Angeles County |
| **Imagery** | Maxar OpenData satellite (pre-fire + 5 post-fire dates: Jan 10/14/15/18/19 2025) |
| **Grid** | 295 cells × 600 m × 600 m |
| **Buildings detected** | 21,797 (Stage 1 SAM3) across 120 cells; 175 cells had no detectable structures |
| **Damage method** | **M2b**: coverage-aware majority vote across valid post-disaster dates |
| **Model** | Same ensemble as Part 1 — zero-shot transfer (trained on xBD floods, applied to wildfire) |

> **Note:** Results are loaded from pre-computed files bundled in this repository (`la_fire_results/`).
> No inference is re-run in this section.


## 2.1  Load Pre-Computed Results

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

# la_fire_results/ is bundled at the same level as this notebook
LA_FIRE_DIR = PACKAGE_ROOT / "la_fire_results"
assert LA_FIRE_DIR.exists(), f"la_fire_results/ not found at {LA_FIRE_DIR}"

# Load sample: top-5 most-damaged cells (M2b), 2,067 buildings
df_la = pd.read_csv(LA_FIRE_DIR / "sample_buildings.csv")
print(f"Sample dataset: {len(df_la):,} buildings across {df_la['cell_id'].nunique()} cells")
print(f"Cells: {sorted(df_la['cell_id'].unique())}")


## 2.2  M2b Damage Distribution

Damage classes assigned by coverage-aware majority vote (M2b) — the authoritative aggregation method.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Full-dataset summary (from damage_summary.md) ──────────────────────────
full_counts = {"No damage": 16246, "Minor": 4174, "Major": 39, "Destroyed": 93, "Unknown": 1245}
total = sum(full_counts.values())
print("Full dataset (21,797 buildings, M2b):")
for cls, n in full_counts.items():
    print(f"  {cls:<12} {n:>6,}  ({n/total*100:.1f}%)")

print()

# ── Sample subset (5 high-damage cells) ────────────────────────────────────
print("Sample subset (5 highest-damage cells, M2b):")
sample_counts = df_la["m2b_damage_label"].value_counts()
print(sample_counts.to_string())

# ── Side-by-side bar charts ─────────────────────────────────────────────────
COLORS = {"no_damage": "#2ecc71", "minor": "#f39c12", "major": "#e67e22", "destroyed": "#e74c3c", "unknown": "#95a5a6"}
LABELS = {"no_damage": "No damage", "minor": "Minor", "major": "Major", "destroyed": "Destroyed", "unknown": "Unknown"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("#1a1a2e")

for ax, (title, counts) in zip(axes, [
    ("Full LA Fire Dataset\n(21,797 buildings)", full_counts),
    ("Sample: 5 High-Damage Cells\n(2,067 buildings)", sample_counts),
]):
    if hasattr(counts, "items"):
        keys = list(counts.keys())
        vals = [counts[k] for k in keys]
        colors = [COLORS.get(k.lower().replace(" ","_"), "#aaa") for k in keys]
        labels = [LABELS.get(k.lower().replace(" ","_"), k) for k in keys]
    else:
        keys = counts.index.tolist()
        vals = counts.values.tolist()
        colors = [COLORS.get(k, "#aaa") for k in keys]
        labels = [LABELS.get(k, k) for k in keys]
    bars = ax.bar(labels, vals, color=colors, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f"{v:,}", ha="center", va="bottom", color="white", fontsize=9)
    ax.set_facecolor("#1a1a2e")
    ax.tick_params(colors="white")
    ax.set_title(title, color="white", fontsize=11)
    ax.set_ylabel("Buildings", color="white")
    for spine in ax.spines.values():
        spine.set_edgecolor("#444")

plt.tight_layout()
plt.show()


## 2.3  Spatial Maps (Eastern LA — Altadena / Eaton Fire Area)

Four map types generated from the full 21,797-building dataset using M2b damage labels.


In [ ]:
from IPython.display import Image, display

map_dir = LA_FIRE_DIR / "maps"
maps_to_show = [
    ("building_damage_map_east.png",   "Building Damage Map — per building polygon, colored by M2b class"),
    ("damage_density_map_east.png",    "Damage Density — spatial hexbin concentration + major/destroyed overlay"),
    ("per_cell_damage_pct_east.png",   "Per-Cell Damage % — circle per 600m cell, color = % minor+major"),
    ("uncertainty_map_east.png",       "Prediction Stability — red = conflicting labels across post dates"),
]

for fname, caption in maps_to_show:
    print(f"\n{caption}")
    display(Image(filename=str(map_dir / fname), width=850))


## 2.4  Multi-Date Temporal Analysis

Each building was assessed across up to 5 post-disaster acquisition dates.
Buildings where predictions changed across dates are flagged as *unstable* —
a signal of domain mismatch (flood-trained model applied to wildfire) rather than genuine ambiguity.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("#1a1a2e")

# Panel 1: valid coverage distribution
ax = axes[0]
cov_counts = df_la["n_dates_valid_coverage"].value_counts().sort_index()
ax.bar(cov_counts.index.astype(str), cov_counts.values,
       color="#3498db", edgecolor="white", linewidth=0.5)
ax.set_facecolor("#1a1a2e")
ax.tick_params(colors="white")
ax.set_title("Valid Post-Date Coverage\nper Building (sample cells)", color="white", fontsize=11)
ax.set_xlabel("Dates with valid imagery", color="white")
ax.set_ylabel("Buildings", color="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444")

# Panel 2: stability pie
ax = axes[1]
stable   = (~df_la["is_unstable"]).sum()
unstable = df_la["is_unstable"].sum()
ax.pie([stable, unstable],
       labels=["Stable\n(consistent across dates)", "Unstable\n(conflicting predictions)"],
       colors=["#2ecc71", "#e74c3c"], autopct="%1.1f%%",
       textprops={"color": "white"}, startangle=90)
ax.set_facecolor("#1a1a2e")
ax.set_title("Prediction Stability\n(sample cells)", color="white", fontsize=11)

plt.tight_layout()
plt.show()

print(f"Sample: {stable:,} stable ({stable/len(df_la)*100:.1f}%), "
      f"{unstable:,} unstable ({unstable/len(df_la)*100:.1f}%)")
print("(51% unstable across full dataset — reflects domain mismatch, not annotation error)")


## 2.5  Spatial Footprint (Sample Cells GeoJSON)

Building polygons for the 5 highest-damage cells, colored by M2b damage class.
Data is in WGS84 GeoJSON — a standard interoperable format (FAIR: Interoperable).


In [ ]:
import geopandas as gpd

DAMAGE_COLORS = {
    "no_damage": "#2ecc71",
    "minor":     "#f39c12",
    "major":     "#e67e22",
    "destroyed": "#e74c3c",
    "unknown":   "#95a5a6",
}

gdf = gpd.read_file(LA_FIRE_DIR / "sample_cells.geojson")
gdf["color"] = gdf["m2b_damage_label"].map(DAMAGE_COLORS).fillna("#aaa")

fig, ax = plt.subplots(figsize=(12, 9))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#1a1a2e")

for label, color in DAMAGE_COLORS.items():
    sub = gdf[gdf["m2b_damage_label"] == label]
    if len(sub):
        sub.plot(ax=ax, color=color, linewidth=0.2, edgecolor="white",
                 alpha=0.85, label=label.replace("_", " ").title())

legend = ax.legend(title="M2b Damage Class", facecolor="#333",
                   labelcolor="white", title_fontsize=10, fontsize=9,
                   loc="lower left")
legend.get_title().set_color("white")
ax.set_title(f"Building Damage — 5 High-Damage Cells ({len(gdf):,} buildings, M2b)",
             color="white", fontsize=12, pad=12)
ax.tick_params(colors="white")
ax.set_xlabel("Longitude", color="white")
ax.set_ylabel("Latitude", color="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444")
plt.tight_layout()
plt.show()

print(f"CRS: {gdf.crs}  |  Columns: {list(gdf.columns)}")


## 2.6  Responsible Use: Domain Mismatch

The Stage-2b ensemble was **trained on flood events** from the xBD benchmark and applied
**zero-shot** to the 2025 LA wildfires. Key implications:

- **"Minor" damage** in wildfire context likely captures discoloration, ash deposits, and roof
  texture changes visible in satellite imagery — not the same as flood-inundated structure.
- **High inter-date instability (51%)** is expected: the model was not calibrated for wildfire
  spectral signatures, leading to inconsistent predictions across acquisition dates.
- **No model retraining** was performed. All results should be interpreted as a *baseline*
  capability demonstration, not a validated operational product.
- Future work: fine-tuning on wildfire-labeled imagery (e.g., FEMA damage surveys + Maxar imagery)
  would substantially improve calibration.


---

# Part 3 — Stage 1 Benchmark: Building Detection on xView2

Stage 1 (SAM3 text-prompted segmentation) was evaluated on the full xView2 challenge benchmark —
933 pre-disaster images across 8 disaster types. This quantifies detection accuracy independently
of the damage classification stages.


In [ ]:
with open(LA_FIRE_DIR / "xview2_eval" / "eval_test.json") as f:
    ev = json.load(f)

print("Stage 1 — SAM3 Building Detection on xView2 Test Set")
print("=" * 52)
print(f"  Images evaluated : {ev['images']:,}")
print(f"  GT buildings     : {ev['gt_total']:,}")
print(f"  Predicted        : {ev['pred_total']:,}")
print(f"  True Positives   : {ev['tp']:,}")
print(f"  IoU threshold    : {ev['iou_thresh']}")
print()
print(f"  Precision : {ev['precision']:.3f}")
print(f"  Recall    : {ev['recall']:.3f}")
print(f"  F1        : {ev['f1']:.3f}")
print(f"  Mean IoU  : {ev['mean_iou_matched']:.3f}  (matched pairs only)")
print()
print("Interpretation:")
print("  High precision (0.682) → when SAM3 finds a building, it is usually correct.")
print("  Lower recall (0.284)  → SAM3 misses many buildings, especially in dense/rural scenes.")
print("  High IoU (0.759)      → matched polygons are geometrically accurate.")


In [ ]:
# Per-disaster breakdown
print(f"{'Disaster':<28} {'Images':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'IoU':>6}")
print("-" * 62)
for name, d in sorted(ev["per_disaster"].items(),
                       key=lambda x: x[1]["f1"], reverse=True):
    print(f"{name:<28} {d['images']:>6} {d['precision']:>6.3f} "
          f"{d['recall']:>6.3f} {d['f1']:>6.3f} {d['mean_iou_matched']:>6.3f}")

print()
print("Note: santa-rosa-wildfire F1=0.547 is the most relevant benchmark")
print("for the LA fire application (same hazard type).")


---

# Part 4 — FAIR Data Principles

This submission was designed to adhere to FAIR data principles as required by the
I-GUIDE Spatial AI Challenge 2025-26.

| Principle | Implementation |
|---|---|
| **Findable** | Code hosted on GitHub (`jikun-tamu/cybertraining-team4`). All data files bundled in the repository or generated reproducibly by running this notebook. |
| **Accessible** | Notebook runs end-to-end from a single `git clone` + `git lfs pull`. No proprietary data required. Input imagery from Maxar OpenData (public). xBD training data publicly available from IEEE GRSS. |
| **Interoperable** | All outputs in open standard formats: GeoJSON (RFC 7946), GeoPackage (OGC standard), CSV. Damage classes follow the xView2 taxonomy (0=no damage, 1=minor, 2=major, 3=destroyed). Building polygons in WGS84 (EPSG:4326). |
| **Reusable** | Pipeline is modular: Stage 1 (detection), Stage 2a (exposure), Stage 2b (damage) can each be run independently. M2b aggregation method documented in `reports/m2b_validation.md`. Domain mismatch limitations explicitly stated. |

## Reproducibility Checklist

To reproduce this notebook from scratch on a new machine:

```bash
# 1. Clone the repository
git clone https://github.com/jikun-tamu/cybertraining-team4
cd cybertraining-team4/II_package/II_package

# 2. Pull model checkpoints (Git LFS)
git lfs pull

# 3. Set your Hugging Face token (for SAM3 gated model access)
export HF_TOKEN=hf_...

# 4. Open and run this notebook
jupyter lab la_fire_challenge_submission.ipynb
```

**Prerequisite**: A Hugging Face account with access to `facebook/sam3`
(accept the model license at huggingface.co/facebook/sam3).

**Runtime**: ~10–15 min on A100 GPU for Part 1 (live pipeline). Parts 2–4 load
pre-computed data and complete in under 1 minute.
